# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates parsing raw synthetic EDI lines and Debezium JSON records into structured Bronze output in the local Iceberg table.

In [ ]:
import os
import json
import uuid
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, udf, to_json, struct, col
from pyspark.sql.types import StringType

# Set up local Spark session
spark = SparkSession.builder \
    .appName("Day03_Batch_and_CDC") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("Spark Session initialized.")

## Simulate X12 Batch Parsing

In [ ]:
# Synthetic X12 837 data
x12_data = [("ISA*00*          *00*          *ZZ*1234567890     *ZZ*999999999      *230501*1234*^*00501*000000001*0*T*:~",),
            ("GS*HC*1234567890*999999999*20230501*1234*1*X*005010X222A1~",)]

df_x12_raw = spark.createDataFrame(x12_data, ["raw_line"])

# Parse UDF (Mock)
def parse_x12(line):
    return json.dumps({"parsed_x12": line.replace("~", "")})

parse_x12_udf = udf(parse_x12, StringType())

df_x12_bronze = df_x12_raw \
    .withColumn("payload_id", udf(lambda: str(uuid.uuid4()), StringType())()) \
    .withColumn("source_system_id", lit("SYNTHETIC_X12")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("raw_payload_json", parse_x12_udf("raw_line")) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

df_x12_bronze.write.format("iceberg").mode("append").save("local.ehdip_bronze.raw_payloads")
print("Synthetic X12 batch appended to Bronze.")

## Simulate CDC Consumption

In [ ]:
# Synthetic Debezium CDC data
cdc_data = [
    ("c", None, '{"id": 1, "status": "NEW"}', {"db": "ehr_prod", "lsn": "1001"}),
    ("u", '{"id": 1, "status": "NEW"}', '{"id": 1, "status": "PROCESSED"}', {"db": "ehr_prod", "lsn": "1002"})
]

df_cdc_raw = spark.createDataFrame(cdc_data, ["op", "before", "after", "source"])

df_cdc_bronze = df_cdc_raw \
    .withColumn("payload_id", col("source.lsn")) \
    .withColumn("source_system_id", col("source.db")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("raw_payload_json", to_json(struct("op", "before", "after", "source"))) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

df_cdc_bronze.write.format("iceberg").mode("append").save("local.ehdip_bronze.raw_payloads")
print("Synthetic CDC batch appended to Bronze.")

In [ ]:
# Verify ingestion
spark.sql("SELECT source_system_id, raw_payload_json FROM local.ehdip_bronze.raw_payloads WHERE source_system_id IN ('SYNTHETIC_X12', 'ehr_prod')").show(truncate=False)